# Searching arXiv papers

This notebook collects papers, pulls a few keywords from their PDFs, and builds an index for finding related work. Generated files go in the project's `data/` folder.

## Setup

Install the required packages and connect Google Drive to access Colab Secrets.

In [ ]:
%pip install -q requests beautifulsoup4 yake pypdf sentence-transformers faiss-cpu llama-index-llms-groq

from google.colab import drive
drive.mount("/content/drive")

## Settings

Choose the topics to search and set how many results to collect.

In [ ]:
from pathlib import Path

PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
INDEX_DIR = DATA_DIR / "index"
for directory in (RAW_DIR, PROCESSED_DIR, INDEX_DIR):
    directory.mkdir(parents=True, exist_ok=True)

SCRAPED_CSV = RAW_DIR / "merged_all.csv"
KEYWORDS_CSV = PROCESSED_DIR / "keyphrases_extracted.csv"
CLEANED_CSV = PROCESSED_DIR / "merged_clean_final.csv"
EMBEDDINGS_NPY = PROCESSED_DIR / "article_embeddings.npy"
VECTORS_CSV = PROCESSED_DIR / "keyphrases_vectors_FINAL.csv"
METADATA_JSON = PROCESSED_DIR / "article_metadata.json"
FAISS_INDEX = INDEX_DIR / "faiss_index.bin"
FAISS_URLS = INDEX_DIR / "faiss_urls.pkl"

TOPICS = [
    "machine learning",
    "natural language processing",
    "computer vision",
]
RESULTS_PER_TOPIC = 200
SCRAPER_WORKERS = 3
PDF_WORKERS = 8
PDF_MAX_PAGES = 2
TOP_KEYPHRASES = 5

EMBEDDING_MODEL = "all-MiniLM-L6-v2"
EMBEDDING_BATCH_SIZE = 256
SEARCH_TOP_K = 3
QUERY = ""
ENABLE_GROQ = True

## Imports and shared helpers

In [ ]:
import csv
import io
import json
import re
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from functools import lru_cache
from urllib.parse import urljoin

import faiss
import pickle
import numpy as np
import pandas as pd
import requests
import yake
from bs4 import BeautifulSoup
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm

HEADERS = {"User-Agent": "semantic-arxiv-colab/1.0"}

def require_file(path):
    if not path.exists():
        raise FileNotFoundError(
            f"Fichier introuvable : {path}. Exécutez l’étape précédente."
        )

def request_url(url, timeout, attempts=3):
    for attempt in range(attempts):
        try:
            response = requests.get(url, headers=HEADERS, timeout=timeout)
            response.raise_for_status()
            return response
        except requests.RequestException:
            if attempt == attempts - 1:
                raise
            time.sleep(2 ** attempt)

@lru_cache(maxsize=1)
def get_embedding_model(model_name=EMBEDDING_MODEL):
    return SentenceTransformer(model_name)

## Collect papers

In [ ]:
ARTICLE_COLUMNS = [
    "topic",
    "title",
    "authors",
    "publication_date",
    "pdf_url",
]

def scrape_arxiv_page(topic, start, limit):
    query = requests.utils.quote(topic)
    url = (
        f"https://arxiv.org/search/?query={query}"
        "&searchtype=all&abstracts=show"
        "&order=-announced_date_first&size=200"
        f"&start={start}"
    )
    response = request_url(url, timeout=30)
    soup = BeautifulSoup(response.text, "html.parser")
    rows = []

    for article in soup.select("li.arxiv-result"):
        title_node = article.select_one("p.title")
        pdf_node = article.select_one("p.list-title a[href*='/pdf/']")
        if title_node is None or pdf_node is None:
            continue

        date_node = article.select_one("p.is-size-7")
        date_match = (
            re.search(
                r"Submitted\s+(\d{1,2}\s+\w+,\s+\d{4})",
                date_node.get_text(" ", strip=True),
            )
            if date_node
            else None
        )
        rows.append(
            {
                "topic": topic,
                "title": title_node.get_text(" ", strip=True),
                "authors": ", ".join(
                    node.get_text(" ", strip=True)
                    for node in article.select("p.authors a")
                ),
                "publication_date": (
                    date_match.group(1) if date_match else ""
                ),
                "pdf_url": urljoin("https://arxiv.org", pdf_node["href"]),
            }
        )

    return rows[:limit]

def scrape_arxiv(
    topics=TOPICS,
    results_per_topic=RESULTS_PER_TOPIC,
    output_path=SCRAPED_CSV,
):
    if not topics:
        raise ValueError("TOPICS ne peut pas être vide.")
    if results_per_topic <= 0:
        raise ValueError("RESULTS_PER_TOPIC doit être supérieur à zéro.")

    tasks = [
        (topic, start, min(200, results_per_topic - start))
        for topic in topics
        for start in range(0, results_per_topic, 200)
    ]
    written = 0

    with output_path.open("w", encoding="utf-8", newline="") as output:
        writer = csv.DictWriter(
            output,
            fieldnames=ARTICLE_COLUMNS,
            delimiter=";",
        )
        writer.writeheader()

        with ThreadPoolExecutor(max_workers=SCRAPER_WORKERS) as executor:
            futures = {
                executor.submit(scrape_arxiv_page, *task): task
                for task in tasks
            }
            for future in tqdm(
                as_completed(futures),
                total=len(futures),
                desc="Pages arXiv",
            ):
                topic, start, _ = futures[future]
                try:
                    rows = future.result()
                except Exception as error:
                    print(
                        f"[Page ignorée] {topic}, départ {start}: {error}"
                    )
                    continue
                writer.writerows(rows)
                written += len(rows)

    if written == 0:
        raise RuntimeError("Aucun article n’a été collecté.")
    print(f"{written} articles enregistrés dans {output_path}")

## Run the collection

In [ ]:
scrape_arxiv()

## Extract keywords from PDFs

In [ ]:
def read_pdf_text(url, max_pages=PDF_MAX_PAGES):
    try:
        response = request_url(url, timeout=60)
        reader = PdfReader(io.BytesIO(response.content))
        return " ".join(
            page.extract_text() or ""
            for page in reader.pages[:max_pages]
        ).strip()
    except Exception as error:
        print(f"[PDF ignoré] {url}: {error}")
        return ""

def process_article(row):
    row = dict(row)
    row["keyphrases_extracted"] = ""
    pdf_url = row.get("pdf_url", "").strip()
    if not pdf_url:
        return row

    text = read_pdf_text(pdf_url)
    if not text:
        return row

    extractor = yake.KeywordExtractor(
        lan="en",
        n=3,
        dedupLim=0.9,
        top=20,
    )
    phrases = [
        phrase
        for phrase, _ in extractor.extract_keywords(text)
        if len(phrase.split()) >= 2
    ]
    row["keyphrases_extracted"] = ", ".join(
        phrases[:TOP_KEYPHRASES]
    )
    return row

def extract_pdf_keywords(
    input_path=SCRAPED_CSV,
    output_path=KEYWORDS_CSV,
):
    require_file(input_path)

    with input_path.open(
        encoding="utf-8",
        newline="",
    ) as source:
        reader = csv.DictReader(source, delimiter=";")
        rows = list(reader)
        input_columns = reader.fieldnames or []

    if not rows:
        raise ValueError("Aucun article dans le fichier collecté.")
    if "pdf_url" not in input_columns:
        raise ValueError("La colonne pdf_url est absente.")

    output_columns = [
        *input_columns,
        "keyphrases_extracted",
    ]
    with ThreadPoolExecutor(max_workers=PDF_WORKERS) as executor:
        processed = executor.map(process_article, rows)
        with output_path.open(
            "w",
            encoding="utf-8",
            newline="",
        ) as output:
            writer = csv.DictWriter(
                output,
                fieldnames=output_columns,
                delimiter=";",
            )
            writer.writeheader()
            for row in tqdm(
                processed,
                total=len(rows),
                desc="PDF traités",
            ):
                writer.writerow(row)

    print(f"Mots-clés enregistrés dans {output_path}")

## Run keyword extraction

In [ ]:
extract_pdf_keywords()

## Clean the data

In [ ]:
def clean_articles(
    input_path=KEYWORDS_CSV,
    output_path=CLEANED_CSV,
):
    require_file(input_path)
    dataframe = pd.read_csv(
        input_path,
        sep=";",
        encoding="utf-8",
        on_bad_lines="skip",
    )

    required = {"pdf_url", "keyphrases_extracted"}
    missing = required.difference(dataframe.columns)
    if missing:
        raise ValueError(f"Colonnes manquantes : {sorted(missing)}")

    dataframe = dataframe.loc[
        :,
        ~dataframe.columns.str.startswith("Unnamed"),
    ]
    dataframe = dataframe.dropna(subset=list(required))
    for column in required:
        dataframe[column] = dataframe[column].astype(str).str.strip()

    dataframe = dataframe[
        (dataframe["pdf_url"] != "")
        & (dataframe["keyphrases_extracted"] != "")
    ]
    dataframe = dataframe.drop_duplicates(
        subset="pdf_url",
    ).reset_index(drop=True)

    if dataframe.empty:
        raise ValueError("Aucun article exploitable après nettoyage.")

    dataframe.to_csv(
        output_path,
        sep=";",
        index=False,
        encoding="utf-8",
    )
    print(f"{len(dataframe)} articles enregistrés dans {output_path}")

## Save the cleaned data

In [ ]:
clean_articles()

## Create embeddings

In [ ]:
METADATA_COLUMNS = [
    "topic",
    "title",
    "authors",
    "publication_date",
    "pdf_url",
]

def create_embeddings(
    input_path=CLEANED_CSV,
    embeddings_path=EMBEDDINGS_NPY,
    metadata_path=METADATA_JSON,
):
    require_file(input_path)
    dataframe = pd.read_csv(
        input_path,
        sep=";",
        usecols=[
            *METADATA_COLUMNS,
            "keyphrases_extracted",
        ],
        na_filter=False,
    )
    if dataframe.empty:
        raise ValueError("Aucun article à vectoriser.")

    model = get_embedding_model()
    vectors = model.encode(
        dataframe["keyphrases_extracted"].tolist(),
        batch_size=EMBEDDING_BATCH_SIZE,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=True,
    )
    vectors = np.ascontiguousarray(
        vectors,
        dtype=np.float32,
    )

    if vectors.ndim != 2 or not np.isfinite(vectors).all():
        raise ValueError("Les embeddings générés sont invalides.")

    metadata = (
        dataframe[METADATA_COLUMNS]
        .fillna("")
        .astype(str)
        .to_dict("records")
    )
    if len(metadata) != vectors.shape[0]:
        raise ValueError("Métadonnées et embeddings désalignés.")

    np.save(embeddings_path, vectors)
    pd.DataFrame(vectors).to_csv(
        VECTORS_CSV,
        index=False,
        header=False,
    )
    with metadata_path.open("w", encoding="utf-8") as output:
        json.dump(
            metadata,
            output,
            ensure_ascii=False,
            indent=2,
        )

    print(
        f"{vectors.shape[0]} embeddings de dimension "
        f"{vectors.shape[1]} enregistrés"
    )

## Save the embeddings

In [ ]:
create_embeddings()

## Build the FAISS index

In [ ]:
def build_faiss_index(
    embeddings_path=EMBEDDINGS_NPY,
    metadata_path=METADATA_JSON,
    index_path=FAISS_INDEX,
):
    require_file(embeddings_path)
    require_file(metadata_path)

    vectors = np.load(
        embeddings_path,
        allow_pickle=False,
    )
    vectors = np.ascontiguousarray(
        vectors,
        dtype=np.float32,
    )
    with metadata_path.open(encoding="utf-8") as source:
        metadata = json.load(source)

    if vectors.ndim != 2 or vectors.shape[0] == 0:
        raise ValueError("Les embeddings doivent être un tableau 2D non vide.")
    if len(metadata) != vectors.shape[0]:
        raise ValueError("Métadonnées et embeddings désalignés.")
    if not np.isfinite(vectors).all():
        raise ValueError("Les embeddings contiennent NaN ou infini.")
    if not vectors.flags.c_contiguous:
        raise ValueError("Les embeddings ne sont pas contigus.")

    faiss.normalize_L2(vectors)
    index = faiss.IndexFlatIP(vectors.shape[1])
    index.add(vectors)
    faiss.write_index(index, str(index_path))
    with FAISS_URLS.open("wb") as output:
        pickle.dump(
            [str(row["pdf_url"]).strip() for row in metadata],
            output,
        )

    print(f"Index FAISS créé : {index.ntotal} vecteurs")
    return index

## Save the index

In [ ]:
build_faiss_index()

## Search for papers

In [ ]:
def search_articles(
    query,
    top_k=SEARCH_TOP_K,
    index_path=FAISS_INDEX,
    metadata_path=METADATA_JSON,
):
    if not query.strip():
        raise ValueError("La requête ne peut pas être vide.")
    if top_k <= 0:
        raise ValueError("top_k doit être supérieur à zéro.")

    require_file(index_path)
    require_file(metadata_path)

    index = faiss.read_index(str(index_path))
    with metadata_path.open(encoding="utf-8") as source:
        metadata = json.load(source)

    if index.ntotal != len(metadata):
        raise ValueError("Index et métadonnées incompatibles.")

    query_vector = get_embedding_model().encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True,
    )
    query_vector = np.ascontiguousarray(
        query_vector,
        dtype=np.float32,
    )
    if query_vector.shape[1] != index.d:
        raise ValueError("Dimension de requête incompatible avec l’index.")

    scores, indices = index.search(
        query_vector,
        min(top_k, index.ntotal),
    )
    results = [
        {
            **metadata[index_id],
            "score": float(score),
        }
        for score, index_id in zip(scores[0], indices[0])
        if index_id >= 0
    ]

    for rank, result in enumerate(results, start=1):
        print(
            f"{rank}. {result['title']}\n"
            f"   {result['pdf_url']}\n"
            f"   similarité : {result['score']:.3f}"
        )
    return results

## Run a search

Set `QUERY` in the settings cell, then run this cell.

In [ ]:
if QUERY.strip():
    results = search_articles(QUERY)
else:
    results = []
    print("Ajoutez une requête dans QUERY avant d’exécuter cette cellule.")

## Summarize the results with Groq

To get a short explanation for each result, add `GROQ_API_KEY` to Colab Secrets. Set `QUERY`, then run the search and analysis cells.

In [ ]:
def explain_results(
    query,
    results,
    max_results=3,
    model_name="openai/gpt-oss-120b",
):
    from google.colab import userdata
    from llama_index.llms.groq import Groq

    api_key = userdata.get("GROQ_API_KEY")
    if not api_key:
        raise RuntimeError(
            "Ajoutez GROQ_API_KEY dans les Secrets Colab."
        )

    llm = Groq(
        model=model_name,
        temperature=0.1,
        api_key=api_key,
    )
    explanations = []

    for result in results[:max_results]:
        text = read_pdf_text(result["pdf_url"])
        if not text:
            explanation = "Le PDF n’a pas pu être lu."
        else:
            prompt = (
                "Explique en deux phrases pourquoi cet article répond "
                f"à la requête « {query} ».\n\n"
                f"Extrait du PDF :\n{text[:4000]}"
            )
            explanation = llm.complete(prompt).text.strip()

        explanations.append(
            {
                **result,
                "explanation": explanation,
            }
        )

    return explanations

if ENABLE_GROQ and QUERY.strip() and results:
    explanations = explain_results(QUERY, results)
    for item in explanations:
        print(item["title"])
        print(item["explanation"], "\n")

## Generated files

The pipeline writes these files under `data/`:

- `raw/merged_all.csv`: collected papers
- `processed/keyphrases_extracted.csv`: papers with extracted keywords
- `processed/merged_clean_final.csv`: cleaned metadata
- `processed/keyphrases_vectors_FINAL.csv`: embedding vectors
- `index/faiss_index.bin` and `index/faiss_urls.pkl`: search index and paper URLs

The intermediate embedding and metadata files are also kept in `processed/` so the index can be rebuilt.